# Chat Anonymizer

Anonimizza i file di chat in `Chats/` usando Qwen3-32B con finestra scorrevole.

**Sostituzioni applicate:**
- Nomi reali di persone → `Persona_N`
- Nomi di aziende / clienti → `Azienda_N`
- Codici identificativi (pallet ID, LU-ID, dialog ref tipo MF12345, ecc.) → `COD_N`
- Prezzi e importi monetari → `PREZZO_N`
- `wamas` / `WAMAS` → `wms` (post-processing regex)

**Output:** file con suffisso `_anon.txt` nella stessa cartella dell'input.

In [1]:
from pydantic import BaseModel, Field
from typing import List
from llama_index.llms.openai_like import OpenAILike
from llama_index.core import PromptTemplate
import os, re
from dotenv import load_dotenv

load_dotenv()

llm = OpenAILike(
    model=os.getenv("MODEL_NAME", "Qwen/Qwen3-32B-AWQ"),
    api_base=os.getenv("VLLM_API_BASE_URL", "http://10.1.2.98/v1"),
    api_key="null",
    is_chat_model=True,
    is_function_calling_model=True,
    timeout=120.0,
    temperature=0.1,
    context_window=12288,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)


class EntityPair(BaseModel):
    original: str = Field(description="Testo originale da anonimizzare")
    replacement: str = Field(description="Sostituto anonimizzato assegnato")


class AnonymizedChunk(BaseModel):
    anonymized_text: str = Field(
        description="Testo del chunk con tutte le entità sensibili già sostituite"
    )
    new_entities: List[EntityPair] = Field(
        description="Nuove entità trovate in questo chunk, non presenti nella mapping fornita"
    )


print("Setup completato.")

Setup completato.


In [2]:
def load_lines(filepath: str) -> list:
    with open(filepath, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f if line.strip()]


def format_mapping(entity_map: dict) -> str:
    if not entity_map:
        return "Nessuna mappatura esistente."
    return "\n".join(f'  "{k}" → "{v}"' for k, v in entity_map.items())


def get_next_counters(entity_map: dict) -> dict:
    """Calcola il prossimo indice disponibile per ogni categoria."""
    maxima = {"Persona": 0, "Azienda": 0, "COD": 0, "PREZZO": 0}
    for v in entity_map.values():
        for cat in maxima:
            m = re.match(rf"^{cat}_(\d+)$", v)
            if m:
                maxima[cat] = max(maxima[cat], int(m.group(1)))
    return {cat: n + 1 for cat, n in maxima.items()}


def apply_wamas(text: str) -> str:
    return re.sub(r"\bwamas\b", "wms", text, flags=re.IGNORECASE)


print("Helper pronti.")

Helper pronti.


In [3]:
ANON_PROMPT = PromptTemplate(
    "Sei un assistente che anonimizza trascrizioni di chat di supporto tecnico per sistemi warehouse.\n\n"
    "MAPPATURA CORRENTE (usa questi sostituti già assegnati per coerenza):\n"
    "{mapping}\n\n"
    "PROSSIMI INDICI DISPONIBILI: Persona_{next_persona}, Azienda_{next_azienda}, "
    "COD_{next_cod}, PREZZO_{next_prezzo}\n\n"
    "TESTO DA ANONIMIZZARE:\n"
    "{chunk}\n\n"
    "ISTRUZIONI:\n"
    "1. Restituisci il testo completamente anonimizzato in `anonymized_text`.\n"
    "2. Sostituisci:\n"
    "   - Nomi reali di persone (nome, cognome o entrambi) → Persona_N\n"
    "   - Nomi di aziende o clienti reali → Azienda_N\n"
    "   - Codici identificativi: ID numerici lunghi, pallet ID, LU-ID, "
    "     riferimenti dialogo (MF12345, CU384, ecc.) → COD_N\n"
    "   - Prezzi e importi monetari → PREZZO_N\n"
    "3. Usa la MAPPATURA CORRENTE per entità già note (stessa entità → stesso sostituto).\n"
    "4. Per entità NUOVE, assegna il prossimo indice disponibile e aggiungile in `new_entities`.\n"
    "5. NON sostituire:\n"
    "   - Timestamp (date e ore)\n"
    "   - Termini tecnici: AGV, WMS, OIL, JIRA, ITS, TPO, QC, AF, ecc.\n"
    "   - Pseudonimi già anonimi: Persona_N, Azienda_N, Support Technician N, "
    "     Company Employee N, Capoturno Picking Progetto Lavoro, Progetto Lavoro, ecc.\n"
    "   - Nomi di file allegati (es. IMG-20210624-WA0000.jpg)\n"
    "   - Emoji, simboli speciali\n"
    "6. Mantieni la struttura esatta del testo (righe, spaziatura, punteggiatura)."
)

print("Prompt pronto.")

Prompt pronto.


In [4]:
def anonymize_file(filepath: str, output_path: str, window_size: int = 50):
    lines = load_lines(filepath)
    total = len(lines)
    base = os.path.basename(filepath)
    print(f"[{base}] {total} righe caricate")

    entity_map: dict = {}
    anonymized_lines: list = []
    current_idx = 0

    while current_idx < total:
        end_idx = min(current_idx + window_size, total)
        chunk = "\n".join(lines[current_idx:end_idx])
        counters = get_next_counters(entity_map)
        mapping_str = format_mapping(entity_map)

        print(f"  Righe [{current_idx+1}–{end_idx}] ...", end=" ", flush=True)

        try:
            result = llm.structured_predict(
                AnonymizedChunk,
                ANON_PROMPT,
                mapping=mapping_str,
                next_persona=counters["Persona"],
                next_azienda=counters["Azienda"],
                next_cod=counters["COD"],
                next_prezzo=counters["PREZZO"],
                chunk=chunk,
            )
        except Exception as e:
            print(f"ERRORE LLM: {e}")
            anonymized_lines.extend(lines[current_idx:end_idx])
            current_idx = end_idx
            continue

        for pair in result.new_entities:
            if pair.original and pair.replacement and pair.original not in entity_map:
                entity_map[pair.original] = pair.replacement

        anonymized_lines.extend(result.anonymized_text.split("\n"))
        print(f"ok  (+{len(result.new_entities)} nuove entità)")
        current_idx = end_idx

    final_text = "\n".join(anonymized_lines)
    final_text = apply_wamas(final_text)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(final_text)

    print(f"\n  → Salvato in: {output_path}")
    print(f"  → Mappatura totale: {len(entity_map)} entità")
    return entity_map


print("Funzione pronta.")

Funzione pronta.


In [5]:
INPUT_FILE  = "../Chats/ChatSSI_clean.txt"
OUTPUT_FILE = "../Chats/ChatSSI_clean_anon.txt"
WINDOW_SIZE = 50

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}\n")

entity_map = anonymize_file(INPUT_FILE, OUTPUT_FILE, window_size=WINDOW_SIZE)

print("\n--- MAPPATURA FINALE ---")
for orig, repl in sorted(entity_map.items(), key=lambda x: x[1]):
    print(f"  {repr(orig):45s} → {repl}")

Input:  ../Chats/ChatSSI_clean.txt
Output: ../Chats/ChatSSI_clean_anon.txt

[ChatSSI_clean.txt] 21168 righe caricate
  Righe [1–50] ... ok  (+9 nuove entità)
  Righe [51–100] ... ok  (+6 nuove entità)
  Righe [101–150] ... ok  (+0 nuove entità)
  Righe [151–200] ... ok  (+1 nuove entità)
  Righe [201–250] ... ok  (+2 nuove entità)
  Righe [251–300] ... ok  (+15 nuove entità)
  Righe [301–350] ... ok  (+4 nuove entità)
  Righe [351–400] ... ok  (+0 nuove entità)
  Righe [401–450] ... ok  (+1 nuove entità)
  Righe [451–500] ... ok  (+1 nuove entità)
  Righe [501–550] ... ok  (+0 nuove entità)
  Righe [551–600] ... ok  (+0 nuove entità)
  Righe [601–650] ... ok  (+3 nuove entità)
  Righe [651–700] ... ok  (+2 nuove entità)
  Righe [701–750] ... ok  (+8 nuove entità)
  Righe [751–800] ... ok  (+0 nuove entità)
  Righe [801–850] ... ok  (+2 nuove entità)
  Righe [851–900] ... ok  (+0 nuove entità)
  Righe [901–950] ... ok  (+6 nuove entità)
  Righe [951–1000] ... ERRORE LLM: <html>
<head><t

KeyboardInterrupt: 